# Facial Age Group Classification

**Author:** Yaseen Mneimneih

Trains and compares four models that classify a face into one of four age groups (Child, Young, Middle-Aged, Senior): HOG + SVM, HOG + MLP, a CNN, and HOG + LBP + SVM. The final sections compare accuracy, inference speed and model size, and show predictions on test images.

In [ ]:
import os
import cv2
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
from joblib import dump, load
from skimage.feature import hog, local_binary_pattern
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models
import time

# Folder containing CW_Dataset/ (train/ and test/ images with label files).
# Trained models are saved to a Models/ subfolder here.
# In Colab, mount Google Drive first and point this at your folder there.
DATA_ROOT = '.'

Loads full dataset when limit=None

Change limit to load a set amount of images

In [ ]:
def load_dataset_limited(data_dir, target_size=(64, 64), limit=None):
    """
    Loads images and labels from the specified directory.
    Uses a dictionary to map filenames to their labels from a master text file,
    and converts images to RGB format immediately.
    """
    images = []
    labels = []

    folder_name = os.path.basename(data_dir)
    label_file = os.path.join(data_dir, f'{folder_name}_labels.txt')

    label_dict = {}
    if os.path.exists(label_file):
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    label_dict[parts[0]] = int(parts[1])
    else:
        print(f"Error: Could not find {label_file}")

    image_paths = sorted(glob.glob(os.path.join(data_dir, '*.jpg')))

    if limit is not None:
        image_paths = image_paths[:limit]

    for img_path in image_paths:
        base_name = os.path.basename(img_path)

        if base_name in label_dict:
            img = cv2.imread(img_path)
            if img is not None:
                # Convert from OpenCV's default BGR to standard RGB
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # Resize to handle varying resolutions
                img_resized = cv2.resize(img, target_size)
                images.append(img_resized)
                labels.append(label_dict[base_name])

    return np.array(images), np.array(labels)

print("Loading the training data...")
X_train, y_train = load_dataset_limited(os.path.join(DATA_ROOT, 'CW_Dataset', 'train'), limit=None)

print("Loading the testing data...")
X_test, y_test = load_dataset_limited(os.path.join(DATA_ROOT, 'CW_Dataset', 'test'), limit=None)

print(f"Loaded {len(X_train)} training images and {len(X_test)} testing images.")

Loading the training data...
Loading the testing data...
Loaded 13300 training images and 850 testing images.


Feature Extraction

In [ ]:
def extract_hog_features(images):
    """Extracts HOG-only features from a list of images."""
    features = []
    for img in images:
        if len(img.shape) == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        else:
            gray = img
        feat = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
        features.append(feat)
    return np.array(features)

print("Extracting HOG-only features for training data...")
X_train_hog = extract_hog_features(X_train)

print("Extracting HOG-only features for testing data...")
X_test_hog = extract_hog_features(X_test)

print("HOG feature extraction complete!")

Extracting HOG-only features for training data...
Extracting HOG-only features for testing data...
HOG feature extraction complete!


Train Model 1 (HOG + SVM)

In [ ]:
print("Training Model 1: HOG + SVM...")
hog_svm = SVC(kernel='rbf', C=1.0, gamma='scale',
              class_weight='balanced', probability=True)
hog_svm.fit(X_train_hog, y_train)

svm_preds = hog_svm.predict(X_test_hog)
print("HOG+SVM Accuracy:", accuracy_score(y_test, svm_preds))
print(classification_report(y_test, svm_preds))

Training Model 1: HOG + SVM...
HOG+SVM Accuracy: 0.7752941176470588
              precision    recall  f1-score   support

           0       0.91      0.93      0.92       150
           1       0.86      0.82      0.84       350
           2       0.61      0.59      0.60       200
           3       0.67      0.76      0.71       150

    accuracy                           0.78       850
   macro avg       0.76      0.78      0.77       850
weighted avg       0.78      0.78      0.78       850



Train Model 2 (HOG+MLP)

In [ ]:
print("Training Model 2: HOG + MLP...")
hog_mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
hog_mlp.fit(X_train_hog, y_train)

mlp_preds = hog_mlp.predict(X_test_hog)
print("HOG+MLP Accuracy:", accuracy_score(y_test, mlp_preds))
print(classification_report(y_test, mlp_preds))

Training Model 2: HOG + MLP...
HOG+MLP Accuracy: 0.7188235294117648
              precision    recall  f1-score   support

           0       0.87      0.83      0.85       150
           1       0.78      0.84      0.80       350
           2       0.59      0.47      0.52       200
           3       0.59      0.65      0.62       150

    accuracy                           0.72       850
   macro avg       0.71      0.70      0.70       850
weighted avg       0.71      0.72      0.71       850



Extract HOG + LBP Features (for Models 3 & 4)

In [ ]:
def extract_advanced_features(images):
    """Extracts combined HOG and LBP features with robust preprocessing."""
    combined_features = []

    radius = 3
    n_points = 8 * radius
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    for img in images:
        # 1. Handle colour channels
        if len(img.shape) == 3 and img.shape[2] == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        else:
            gray = img.squeeze()

        # 2. Ensure uint8 for CLAHE
        if gray.dtype != np.uint8:
            if gray.max() <= 1.0:
                gray = (gray * 255).astype(np.uint8)
            else:
                gray = gray.astype(np.uint8)

        # 3. Noise reduction
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)

        # 4. Lighting normalisation
        enhanced = clahe.apply(blurred)

        # 5. HOG features
        hog_feat = hog(enhanced, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)

        # 6. LBP features
        lbp = local_binary_pattern(enhanced, n_points, radius, method='uniform')
        n_bins = int(lbp.max() + 1)
        lbp_hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins))
        lbp_hist = lbp_hist.astype('float')
        lbp_hist /= (lbp_hist.sum() + 1e-7)

        combined = np.hstack((hog_feat, lbp_hist))
        combined_features.append(combined)

    return np.array(combined_features)

print("Extracting HOG+LBP features for training data...")
X_train_adv = extract_advanced_features(X_train)

print("Extracting HOG+LBP features for testing data...")
X_test_adv = extract_advanced_features(X_test)

print("Advanced feature extraction complete!")

Extracting HOG+LBP features for training data...
Extracting HOG+LBP features for testing data...
Advanced feature extraction complete!


Train Model 3 (CNN)

In [ ]:
print("Training Model 3: CNN (with Balanced Class Weights)...")

X_train_cnn = X_train.astype('float32') / 255.0
X_test_cnn = X_test.astype('float32') / 255.0

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=10,
    validation_data=(X_test_cnn, y_test),
    batch_size=32,
    class_weight=class_weight_dict
)

Training Model 3: CNN (with Balanced Class Weights)...
Class weights: {0: np.float64(1.3854166666666667), 1: np.float64(0.6045454545454545), 2: np.float64(1.1083333333333334), 3: np.float64(1.3854166666666667)}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
416/416 ━━━━━━━━━━━━━━━━━━━━ 43s 101ms/step - accuracy: 0.5749 - loss: 0.9820 - val_accuracy: 0.6894 - val_loss: 0.7673
Epoch 2/10
416/416 ━━━━━━━━━━━━━━━━━━━━ 41s 99ms/step - accuracy: 0.6649 - loss: 0.7752 - val_accuracy: 0.7318 - val_loss: 0.6709
Epoch 3/10
416/416 ━━━━━━━━━━━━━━━━━━━━ 42s 102ms/step - accuracy: 0.6970 - loss: 0.7018 - val_accuracy: 0.7424 - val_loss: 0.6320
Epoch 4/10
416/416 ━━━━━━━━━━━━━━━━━━━━ 41s 99ms/step - accuracy: 0.7169 - loss: 0.6469 - val_accuracy: 0.7565 - val_loss: 0.6014
Epoch 5/10
416/416 ━━━━━━━━━━━━━━━━━━━━ 83s 101ms/step - accuracy: 0.7378 - loss: 0.6052 - val_accuracy: 0.7376 - val_loss: 0.6216
Epoch 6/10
416/416 ━━━━━━━━━━━━━━━━━━━━ 81s 99ms/step - accuracy: 0.7486 - loss: 0.5779 - val_accuracy: 0.7765 - val_loss: 0.5985
Epoch 7/10
416/416 ━━━━━━━━━━━━━━━━━━━━ 42s 100ms/step - accuracy: 0.7600 - loss: 0.5383 - val_accuracy: 0.7729 - val_loss: 0.6036
Epoch 8/10
416/416 ━━━━━━━━━━━━━━━━━━━━ 81s 98ms/step - accuracy: 0.7692 - loss: 0.523

Train Model 4 (HOG + LBP + SVM)

In [ ]:
print("Training Model 4: HOG + LBP + SVM...")
final_svm = SVC(kernel='rbf', C=1.0, gamma='scale',
                class_weight='balanced', probability=True)
final_svm.fit(X_train_adv, y_train)

adv_preds = final_svm.predict(X_test_adv)
print("HOG+LBP+SVM Accuracy:", accuracy_score(y_test, adv_preds))
print(classification_report(y_test, adv_preds))

Training Model 4: HOG + LBP + SVM...
HOG+LBP+SVM Accuracy: 0.7176470588235294
              precision    recall  f1-score   support

           0       0.85      0.89      0.87       150
           1       0.82      0.76      0.79       350
           2       0.52      0.53      0.53       200
           3       0.62      0.70      0.66       150

    accuracy                           0.72       850
   macro avg       0.71      0.72      0.71       850
weighted avg       0.72      0.72      0.72       850



Save All Models

In [ ]:
print("Saving all models...")

models_dir = os.path.join(DATA_ROOT, 'Models')
os.makedirs(models_dir, exist_ok=True)

dump(hog_svm, os.path.join(models_dir, 'hog_svm.joblib'))
print("HOG+SVM saved.")

dump(hog_mlp, os.path.join(models_dir, 'hog_mlp.joblib'))
print("HOG+MLP saved.")

cnn_model.save(os.path.join(models_dir, 'cnn_model.h5'))
print("CNN saved.")

dump(final_svm, os.path.join(models_dir, 'advanced_svm.joblib'))
print("HOG+LBP+SVM saved.")

print("All models saved successfully!")

Saving all models...
HOG+SVM saved.


HOG+MLP saved.
CNN saved.
HOG+LBP+SVM saved.
All models saved successfully!


Quantitative comparison table

In [ ]:
print("Generating quantitative comparison table...\n")

# Inference times (ms per image)
start = time.time()
hog_svm.predict(X_test_hog)
svm_time = (time.time() - start) / len(X_test_hog) * 1000

start = time.time()
hog_mlp.predict(X_test_hog)
mlp_time = (time.time() - start) / len(X_test_hog) * 1000

start = time.time()
cnn_model.predict(X_test_cnn, verbose=0)
cnn_time = (time.time() - start) / len(X_test_cnn) * 1000

start = time.time()
final_svm.predict(X_test_adv)
adv_svm_time = (time.time() - start) / len(X_test_adv) * 1000

# Model sizes (KB)
svm_size = os.path.getsize(os.path.join(models_dir, 'hog_svm.joblib')) / 1024
mlp_size = os.path.getsize(os.path.join(models_dir, 'hog_mlp.joblib')) / 1024
cnn_size = os.path.getsize(os.path.join(models_dir, 'cnn_model.h5')) / 1024
adv_size = os.path.getsize(os.path.join(models_dir, 'advanced_svm.joblib')) / 1024

# Accuracies
svm_acc     = accuracy_score(y_test, hog_svm.predict(X_test_hog))
mlp_acc     = accuracy_score(y_test, hog_mlp.predict(X_test_hog))
cnn_acc     = accuracy_score(y_test, cnn_model.predict(X_test_cnn, verbose=0).argmax(axis=1))
adv_svm_acc = accuracy_score(y_test, final_svm.predict(X_test_adv))

print(f"{'Model':<25} {'Accuracy':>10} {'Inference(ms)':>15} {'Size(KB)':>10}")
print("-" * 65)
print(f"{'HOG + SVM':<25} {svm_acc:>10.3f} {svm_time:>15.4f} {svm_size:>10.1f}")
print(f"{'HOG + MLP':<25} {mlp_acc:>10.3f} {mlp_time:>15.4f} {mlp_size:>10.1f}")
print(f"{'CNN':<25} {cnn_acc:>10.3f} {cnn_time:>15.4f} {cnn_size:>10.1f}")
print(f"{'HOG + LBP + SVM':<25} {adv_svm_acc:>10.3f} {adv_svm_time:>15.4f} {adv_size:>10.1f}")

Generating quantitative comparison table...

Model                       Accuracy   Inference(ms)   Size(KB)
-----------------------------------------------------------------
HOG + SVM                      0.775         14.0085   133384.1
HOG + MLP                      0.719          0.0122     5540.1
CNN                            0.759          1.5956     9678.5
HOG + LBP + SVM                0.718         13.3735   142220.1


Qualitative results

In [ ]:
age_groups = {0: 'Child', 1: 'Young', 2: 'Middle-Aged', 3: 'Senior'}
indices = random.sample(range(len(X_test)), 8)

# --- Model 1: HOG + SVM ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Model 1: HOG + SVM — Predictions vs Ground Truth', fontsize=14)
for ax, idx in zip(axes.flat, indices):
    pred = hog_svm.predict(X_test_hog[idx:idx+1])[0]
    true = y_test[idx]
    ax.imshow(X_test[idx])
    ax.set_title(f"Pred: {age_groups[pred]}\nTrue: {age_groups[true]}",
                 color='green' if pred == true else 'red')
    ax.axis('off')
plt.tight_layout()
plt.show()

# --- Model 2: HOG + MLP ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Model 2: HOG + MLP — Predictions vs Ground Truth', fontsize=14)
for ax, idx in zip(axes.flat, indices):
    pred = hog_mlp.predict(X_test_hog[idx:idx+1])[0]
    true = y_test[idx]
    ax.imshow(X_test[idx])
    ax.set_title(f"Pred: {age_groups[pred]}\nTrue: {age_groups[true]}",
                 color='green' if pred == true else 'red')
    ax.axis('off')
plt.tight_layout()
plt.show()

# --- Model 3: CNN ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Model 3: CNN — Predictions vs Ground Truth', fontsize=14)
for ax, idx in zip(axes.flat, indices):
    pred = cnn_model.predict(X_test_cnn[idx:idx+1], verbose=0).argmax()
    true = y_test[idx]
    ax.imshow(X_test[idx])
    ax.set_title(f"Pred: {age_groups[pred]}\nTrue: {age_groups[true]}",
                 color='green' if pred == true else 'red')
    ax.axis('off')
plt.tight_layout()
plt.show()

# --- Model 4: HOG + LBP + SVM ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Model 4: HOG + LBP + SVM — Predictions vs Ground Truth', fontsize=14)
for ax, idx in zip(axes.flat, indices):
    pred = final_svm.predict(X_test_adv[idx:idx+1])[0]
    true = y_test[idx]
    ax.imshow(X_test[idx])
    ax.set_title(f"Pred: {age_groups[pred]}\nTrue: {age_groups[true]}",
                 color='green' if pred == true else 'red')
    ax.axis('off')
plt.tight_layout()
plt.show()